## SOLAR PANEL DEFECT CLASSIFICATION

In [ ]:
pip install tensorflow

: 

In [2]:
import tensorflow as tf
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
data_dir = r"C:\Users\Lenovo\OneDrive\Desktop\Solar Panel Defect\Data"
batch_size = 32
img_height = 180
img_width = 180

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

In [ ]:
class_names= train_ds.class_names
num_classes= len(class_names)

In [ ]:
class_names

## BASE MODEL(CNN 2D)

In [ ]:
model=tf.keras.models.Sequential()
model.add(tf.keras.layers.Rescaling(1.0/255, input_shape=(img_height,img_width,3)))

In [ ]:
## CONVOLUTION LAYER 1
model.add(tf.keras.layers.Conv2D(32, (3,3), activation="relu"))
model.add(tf.keras.layers.MaxPooling2D((2,2)))

## CONVOLUTION LAYER 2
model.add(tf.keras.layers.Conv2D(64, (3,3), activation="relu"))
model.add(tf.keras.layers.MaxPooling2D((2,2)))

## CONVOLUTION LAYER 3
model.add(tf.keras.layers.Conv2D(128, (3,3), activation="relu"))
model.add(tf.keras.layers.MaxPooling2D((2,2)))

##FLATTENING
model.add(tf.keras.layers.Flatten())

##FCL
model.add(tf.keras.layers.Dense(128, activation="relu"))

##OUTPUT LAYER
model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))

In [ ]:
model.compile(optimizer="adam", 
              loss="sparse_categorical_crossentropy",
             metrics=["accuracy"])

In [ ]:
model.summary()

In [ ]:
Epochs=20

In [ ]:
Solar=model.fit(train_ds,
               validation_data=val_ds,
               epochs=Epochs)

In [ ]:
train_accuracy=Solar.history['accuracy']
val_accuracy=Solar.history['val_accuracy']

# Create a range from 1 to the total number of actual training epochs
Epochs = range(1, len(train_accuracy) + 1)

plt.figure(figsize=(10,5))
plt.plot(Epochs, train_accuracy, label='training_accuracy', marker='o')
plt.plot(Epochs, val_accuracy, label='val_accuracy', marker='o')
plt.title('Training Vs Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## SOLVE THE OVERFITTING PROBLEM

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stooping= EarlyStopping(monitor='val_loss', patience=3)

In [ ]:
from tensorflow.keras.callbacks import LearningRateScheduler

def scheduler(epoch, lr):
    if epoch>5:
        return lr*0.1
    return lr

lr_schedule= LearningRateScheduler(scheduler)

In [ ]:
model=tf.keras.models.Sequential()
model.add(tf.keras.layers.Rescaling(1.0/255, input_shape=(img_height,img_width,3)))

## IMAGE AUGMENTATION LAYERS (Only active during model.fit)
model.add(tf.keras.layers.RandomFlip("horizontal_and_vertical"))
model.add(tf.keras.layers.RandomRotation(0.2))
model.add(tf.keras.layers.RandomZoom(0.2))
model.add(tf.keras.layers.RandomContrast(0.2))

## CONVOLUTION LAYER 1
model.add(tf.keras.layers.Conv2D(32, (3,3), activation="relu"))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.MaxPooling2D((2,2)))
model.add(tf.keras.layers.Dropout(0.3))

## CONVOLUTION LAYER 2
model.add(tf.keras.layers.Conv2D(64, (3,3), activation="relu"))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.MaxPooling2D((2,2)))
model.add(tf.keras.layers.Dropout(0.3))

## CONVOLUTION LAYER 3
model.add(tf.keras.layers.Conv2D(128, (3,3), activation="relu"))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.MaxPooling2D((2,2)))
model.add(tf.keras.layers.Dropout(0.3))

##FLATTENING
model.add(tf.keras.layers.GlobalAveragePooling2D())
model.add(tf.keras.layers.Dropout(0.5))

##FCL
model.add(tf.keras.layers.Dense(128, activation="relu"))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dropout(0.5))

##OUTPUT LAYER
model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))

In [ ]:
model.compile(optimizer="adam", 
              loss="sparse_categorical_crossentropy",
             metrics=["accuracy"])

In [ ]:
Epochs=10

In [ ]:
Solar=model.fit(train_ds,
               validation_data=val_ds,
               epochs=Epochs)

## TRANSFER LEARNING- MobileNetV2

In [ ]:
base_model= tf.keras.applications.MobileNetV2(
    input_shape=(180,180,3),
    include_top=False,
    weights='imagenet'
)

In [ ]:
base_model.trainable= False
model= tf.keras.models.Sequential()

model.add(tf.keras.layers.Rescaling(1.0/255, input_shape=(img_height,img_width,3)))

## IMAGE AUGMENTATION LAYERS (Only active during model.fit)
model.add(tf.keras.layers.RandomFlip("horizontal_and_vertical"))
model.add(tf.keras.layers.RandomRotation(0.2))
model.add(tf.keras.layers.RandomZoom(0.2))
model.add(tf.keras.layers.RandomContrast(0.2))

model.add(base_model)
model.add(tf.keras.layers.GlobalAveragePooling2D())

model.add(tf.keras.layers.Dense(128, activation="relu"))
model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))

In [ ]:
model.compile(optimizer="adam", 
              loss="sparse_categorical_crossentropy",
             metrics=["accuracy"])

In [ ]:
model.summary()

In [ ]:
Solar=model.fit(train_ds,
               validation_data=val_ds,
               epochs=10)

In [ ]:
train_accuracy=Solar.history['accuracy']
val_accuracy=Solar.history['val_accuracy']

# Create a range from 1 to the total number of actual training epochs
Epochs = range(1, len(train_accuracy) + 1)

plt.figure(figsize=(10,5))
plt.plot(Epochs, train_accuracy, label='training_accuracy', marker='o')
plt.plot(Epochs, val_accuracy, label='val_accuracy', marker='o')
plt.title('Training Vs Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
model.save("mobilenetv5.keras")

## EFFICIENTNETB0

In [ ]:
from tensorflow.keras.applications.efficientnet import EfficientNetB0

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

In [ ]:
class_names= train_ds.class_names
num_classes= len(class_names)

In [ ]:
class_names

In [ ]:
import os
from keras.applications.resnet50 import ResNet50, preprocess_input

In [ ]:
class_counts={}
total_images=0

for class_name in class_names:
    class_path= os.path.join(data_dir, class_name)
    count= len(os.listdir(class_path))
    class_counts[class_name]= count
    total_images+= count

In [ ]:
class_weights= {}
for index,class_name in enumerate(class_names):
    class_weights[index]= total_images / (num_classes*class_counts[class_name])

print("class_weights",class_weights)

In [ ]:
data_augmentation=tf.keras.models.Sequential()
data_augmentation.add(tf.keras.layers.RandomFlip("horizontal"))
data_augmentation.add(tf.keras.layers.RandomRotation(0.2))
data_augmentation.add(tf.keras.layers.RandomZoom(0.2))

In [ ]:
train_images=[]
train_labels=[]

In [ ]:
for images,labels in train_ds:
    images= preprocess_input(images)
    train_images.append(images)
    train_labels.append(labels)

In [ ]:
train_images= tf.concat(train_images, axis=0)
train_labels= tf.concat(train_labels, axis=0)

In [ ]:
validation_images=[]
validation_labels=[]

In [ ]:
for images,labels in val_ds:
    images= preprocess_input(images)
    validation_images.append(images)
    validation_labels.append(labels)

In [ ]:
validation_images= tf.concat(validation_images, axis=0)
validation_labels= tf.concat(validation_labels, axis=0)

In [ ]:
base_model= tf.keras.applications.EfficientNetB0(
    input_shape=(180,180,3),
    include_top=False,
    weights='imagenet'
)

In [ ]:
base_model.trainable= False
model= tf.keras.models.Sequential()

model.add(tf.keras.layers.Rescaling(1.0/255, input_shape=(img_height,img_width,3)))

## IMAGE AUGMENTATION LAYERS (Only active during model.fit)
model.add(tf.keras.layers.RandomFlip("horizontal_and_vertical"))
model.add(tf.keras.layers.RandomRotation(0.2))
model.add(tf.keras.layers.RandomZoom(0.2))
model.add(tf.keras.layers.RandomContrast(0.2))

model.add(base_model)
model.add(tf.keras.layers.GlobalAveragePooling2D())

model.add(tf.keras.layers.Dense(128, activation="relu"))
model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))

In [ ]:
model.compile(optimizer="adam", 
              loss="sparse_categorical_crossentropy",
             metrics=["accuracy"])

In [ ]:
model.summary()

In [ ]:
Solar=model.fit(train_ds,
               validation_data=val_ds,
               epochs=10)

In [ ]:
model.save('efficient_model.keras')

In [ ]:
import os
print(os.path.exists('efficient_model.keras'))

In [ ]:
loaded_model = tf.keras.models.load_model('efficient_model.keras')